In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q xgboost

import os
import glob
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, recall_score

PROJECT_DIR = '/content/drive/MyDrive/man-bot-detection'
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'notebooks/results')
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)

In [ ]:
DELBOT_PATH = f'{PROJECT_DIR}/data/raw/delbot_mouse'
WINDOW_SIZE = 100
STEP = 50


def normalize_coords(traj):
    if len(traj) == 0:
        return traj
    min_x, max_x = np.min(traj[:, 0]), np.max(traj[:, 0])
    min_y, max_y = np.min(traj[:, 1]), np.max(traj[:, 1])
    range_x = max_x - min_x if max_x - min_x > 0 else 1
    range_y = max_y - min_y if max_y - min_y > 0 else 1
    norm_x = (traj[:, 0] - min_x) / range_x * 200
    norm_y = (traj[:, 1] - min_y) / range_y * 200
    return np.column_stack((norm_x, norm_y))


def load_delbot_trajectory(file_path):
    coords, is_down_flags = [], []
    is_down = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('resolution'):
                continue
            parts = line.split(',')
            if len(parts) < 4:
                continue
            action = parts[1]
            try:
                x = float(parts[2])
                y = float(parts[3])
            except ValueError:
                continue
            if not (np.isfinite(x) and np.isfinite(y)):
                continue
            if action == 'Pressed':
                is_down = 1
            elif action == 'Released':
                is_down = 0
            coords.append((x, y))
            is_down_flags.append(is_down)
    return (np.array(coords, dtype=np.float32),
            np.array(is_down_flags, dtype=np.int32))


def extract_window_features(window_coords, window_clicks=None):
    if len(window_coords) < 5:
        return None
    dx = np.diff(window_coords[:, 0])
    dy = np.diff(window_coords[:, 1])
    speeds = np.sqrt(dx**2 + dy**2)
    if len(speeds) == 0:
        return None
    mean_speed, max_speed, std_speed = np.mean(speeds), np.max(speeds), np.std(speeds)
    path_length = np.sum(speeds)
    num_stops = np.sum(speeds < 0.5)
    angles = np.arctan2(dy, dx)
    if len(angles) > 1:
        angle_diffs = np.abs(np.diff(angles))
        angle_diffs = np.minimum(angle_diffs, 2*np.pi - angle_diffs)
        mean_angle = np.mean(angle_diffs)
    else:
        mean_angle = 0
    num_clicks = 0
    if window_clicks is not None and len(window_clicks) == len(window_coords):
        num_clicks = np.sum((window_clicks[1:] == 1) & (window_clicks[:-1] == 0))
    return {
        'mean_speed': mean_speed, 'max_speed': max_speed, 'std_speed': std_speed,
        'path_length': path_length, 'num_stops': num_stops, 'mean_angle': mean_angle,
        'num_clicks': num_clicks, 'num_points': len(window_coords)
    }


def process_files_finegrained(file_list, label, subtype):
    rows = []
    for file_path in file_list:
        try:
            coords, is_down = load_delbot_trajectory(file_path)
            if len(coords) < WINDOW_SIZE:
                continue
            coords_norm = normalize_coords(coords)
            for start in range(0, len(coords_norm) - WINDOW_SIZE + 1, STEP):
                wc = coords_norm[start:start+WINDOW_SIZE]
                wcl = is_down[start:start+WINDOW_SIZE]
                feats = extract_window_features(wc, wcl)
                if feats is None:
                    continue
                feats['label'] = label
                feats['subtype'] = subtype
                rows.append(feats)
        except Exception:
            continue
    return rows

In [ ]:
all_rows = []
for folder in sorted(glob.glob(f'{DELBOT_PATH}/circles_human_*')):
    subtype = os.path.basename(folder)
    files = glob.glob(f'{folder}/*')
    all_rows += process_files_finegrained(files, 'human', subtype)
    print(subtype, len(files))

for folder in sorted(glob.glob(f'{DELBOT_PATH}/circles_bot_*')):
    subtype = os.path.basename(folder)
    files = glob.glob(f'{folder}/*')
    all_rows += process_files_finegrained(files, 'bot', subtype)
    print(subtype, len(files))

df_behav_fine = pd.DataFrame(all_rows)
print(len(df_behav_fine))
print(df_behav_fine['subtype'].value_counts())

In [ ]:
FEATURE_COLS = ['mean_speed', 'max_speed', 'std_speed', 'path_length',
                 'num_stops', 'mean_angle', 'num_clicks', 'num_points']

rf_behav = joblib.load(f'{PROJECT_DIR}/models/behavioral_rf_300.pkl')

X_fine = df_behav_fine[FEATURE_COLS]
y_fine_true = df_behav_fine['label']
y_fine_pred = rf_behav.predict(X_fine)

cm = confusion_matrix(y_fine_true, y_fine_pred, labels=['human', 'bot'])
print(pd.DataFrame(cm, index=['true_human', 'true_bot'], columns=['pred_human', 'pred_bot']))
print()
print(classification_report(y_fine_true, y_fine_pred, zero_division=0))

df_behav_fine['pred'] = y_fine_pred
df_behav_fine['correct'] = (df_behav_fine['label'] == df_behav_fine['pred'])

subtype_report = (
    df_behav_fine.groupby('subtype')
    .agg(n_windows=('correct', 'size'),
         accuracy=('correct', 'mean'),
         true_label=('label', 'first'))
    .sort_values('accuracy')
)
print(subtype_report.to_string())
subtype_report.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'behavioral_subtype_errors.csv'))

plt.figure(figsize=(10, 5))
colors = ['tab:red' if lbl == 'bot' else 'tab:blue' for lbl in subtype_report['true_label']]
plt.barh(subtype_report.index, subtype_report['accuracy'], color=colors)
plt.xlabel('accuracy')
plt.title('accuracy')
plt.xlim(0, 1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'behavioral_subtype_accuracy.png'), dpi=300)
plt.show()

print(subtype_report[subtype_report['true_label'] == 'bot'].index[0]
      if (subtype_report['true_label'] == 'bot').any() else None)

In [ ]:
DATA_BASE = os.path.join(PROJECT_DIR, 'data/raw')
PATHS = {
    'CIC': os.path.join(DATA_BASE, 'cic_ids_2017'),
    'UNSW': os.path.join(DATA_BASE, 'unsw_nb15'),
}


def load_cic_with_multiclass():
    files = glob.glob(os.path.join(PATHS['CIC'], '*.csv'))
    df = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
    if len(df) > 200000:
        df = df.sample(n=200000, random_state=42)
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    y_multi = df[' Label'].astype(str).str.strip()
    y_binary = (y_multi != 'BENIGN').astype(int)
    X = df.drop(columns=[' Label']).select_dtypes(include=[np.number])
    return X.reset_index(drop=True), y_binary.reset_index(drop=True), y_multi.reset_index(drop=True)


def load_unsw_with_multiclass():
    df_train = pd.read_csv(os.path.join(PATHS['UNSW'], 'UNSW_NB15_training-set.csv'), low_memory=False)
    df_test = pd.read_csv(os.path.join(PATHS['UNSW'], 'UNSW_NB15_testing-set.csv'), low_memory=False)
    df = pd.concat([df_train, df_test], ignore_index=True)
    if len(df) > 200000:
        df = df.sample(n=200000, random_state=42)
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    y_multi = df['attack_cat'].astype(str).str.strip()
    y_binary = (y_multi.str.lower() != 'normal').astype(int)
    X = df.drop(columns=['attack_cat', 'label']).select_dtypes(include=[np.number])
    return X.reset_index(drop=True), y_binary.reset_index(drop=True), y_multi.reset_index(drop=True)


def analyze_network_errors(name, X, y_binary, y_multi):
    print(name)
    X_train, X_test, y_train, y_test, ymulti_train, ymulti_test = train_test_split(
        X, y_binary, y_multi, test_size=0.2, random_state=42, stratify=y_binary
    )
    model = RandomForestClassifier(n_estimators=50, random_state=42,
                                    class_weight='balanced', n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    print(pd.DataFrame(cm, index=['true_0', 'true_1'], columns=['pred_0', 'pred_1']))

    results = []
    for attack_type in sorted(ymulti_test.unique()):
        mask = (ymulti_test == attack_type)
        if mask.sum() == 0:
            continue
        true_label = y_test[mask].iloc[0]
        recall_type = (y_pred[mask.values] == true_label).mean()
        results.append({'attack_type': attack_type, 'n_samples': int(mask.sum()),
                         'is_attack': bool(true_label == 1), 'recall': recall_type})

    df_type_report = pd.DataFrame(results).sort_values('recall')
    print(df_type_report.to_string(index=False))
    df_type_report.to_csv(os.path.join(OUTPUT_DIR, 'tables', f'network_{name}_type_errors.csv'), index=False)

    plt.figure(figsize=(10, 5))
    colors = ['tab:red' if a else 'tab:blue' for a in df_type_report['is_attack']]
    plt.barh(df_type_report['attack_type'], df_type_report['recall'], color=colors)
    plt.xlabel('recall')
    plt.title(name)
    plt.xlim(0, 1)
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'figures', f'network_{name}_type_recall.png'), dpi=300)
    plt.show()

    worst = df_type_report[df_type_report['is_attack']].iloc[0] if df_type_report['is_attack'].any() else None
    if worst is not None:
        print(name, worst['attack_type'], worst['recall'], worst['n_samples'])
    return df_type_report

In [ ]:
X_cic, y_cic_bin, y_cic_multi = load_cic_with_multiclass()
cic_type_report = analyze_network_errors('CIC', X_cic, y_cic_bin, y_cic_multi)

In [ ]:
X_unsw, y_unsw_bin, y_unsw_multi = load_unsw_with_multiclass()
unsw_type_report = analyze_network_errors('UNSW', X_unsw, y_unsw_bin, y_unsw_multi)

In [ ]:
WINDOW_SIZE_SMALL = 20
STEP_SMALL = 10


def process_files_finegrained_small(file_list, label, subtype):
    rows = []
    for file_path in file_list:
        try:
            coords, is_down = load_delbot_trajectory(file_path)
            if len(coords) < WINDOW_SIZE_SMALL:
                continue
            coords_norm = normalize_coords(coords)
            for start in range(0, len(coords_norm) - WINDOW_SIZE_SMALL + 1, STEP_SMALL):
                wc = coords_norm[start:start+WINDOW_SIZE_SMALL]
                wcl = is_down[start:start+WINDOW_SIZE_SMALL]
                feats = extract_window_features(wc, wcl)
                if feats is None:
                    continue
                feats['label'] = label
                feats['subtype'] = subtype
                rows.append(feats)
        except Exception:
            continue
    return rows


all_rows_small = []
for folder in sorted(glob.glob(f'{DELBOT_PATH}/circles_human_*')):
    subtype = os.path.basename(folder)
    files = glob.glob(f'{folder}/*')
    all_rows_small += process_files_finegrained_small(files, 'human', subtype)

for folder in sorted(glob.glob(f'{DELBOT_PATH}/circles_bot_*')):
    subtype = os.path.basename(folder)
    files = glob.glob(f'{folder}/*')
    all_rows_small += process_files_finegrained_small(files, 'bot', subtype)

df_behav_small = pd.DataFrame(all_rows_small)
print(len(df_behav_small))
print(df_behav_small['subtype'].value_counts())

In [ ]:
X_small = df_behav_small[FEATURE_COLS]
y_small = df_behav_small['label']

X_tr_s, X_te_s, y_tr_s, y_te_s, idx_tr_s, idx_te_s = train_test_split(
    X_small, y_small, df_behav_small.index, test_size=0.2, random_state=42, stratify=y_small
)

rf_small = RandomForestClassifier(n_estimators=200, random_state=42,
                                   class_weight='balanced', n_jobs=-1)
rf_small.fit(X_tr_s, y_tr_s)
y_pred_s = rf_small.predict(X_te_s)

cm_s = confusion_matrix(y_te_s, y_pred_s, labels=['human', 'bot'])
print(pd.DataFrame(cm_s, index=['true_human', 'true_bot'], columns=['pred_human', 'pred_bot']))
print()
print(classification_report(y_te_s, y_pred_s, zero_division=0))

test_subtypes = df_behav_small.loc[idx_te_s, 'subtype'].values
df_test_small = pd.DataFrame({
    'subtype': test_subtypes, 'label': y_te_s.values, 'pred': y_pred_s
})
df_test_small['correct'] = (df_test_small['label'] == df_test_small['pred'])

subtype_report_small = (
    df_test_small.groupby('subtype')
    .agg(n_windows=('correct', 'size'), accuracy=('correct', 'mean'),
         true_label=('label', 'first'))
    .sort_values('accuracy')
)
print(subtype_report_small.to_string())
subtype_report_small.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'behavioral_subtype_errors_smallwindow.csv'))

plt.figure(figsize=(10, 5))
colors_s = ['tab:red' if lbl == 'bot' else 'tab:blue' for lbl in subtype_report_small['true_label']]
plt.barh(subtype_report_small.index, subtype_report_small['accuracy'], color=colors_s)
plt.xlabel('accuracy')
plt.title('accuracy')
plt.xlim(0, 1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'behavioral_subtype_accuracy_smallwindow.png'), dpi=300)
plt.show()

if 'circles_bot_gan' in subtype_report_small.index:
    gan_acc = subtype_report_small.loc['circles_bot_gan', 'accuracy']
    gan_n = subtype_report_small.loc['circles_bot_gan', 'n_windows']
    print(gan_n, gan_acc)
else:
    print('circles_bot_gan')

worst_bot_overall = subtype_report_small[subtype_report_small['true_label'] == 'bot'].index[0] \
    if (subtype_report_small['true_label'] == 'bot').any() else None
worst_bot_overall_acc = subtype_report_small[subtype_report_small['true_label'] == 'bot']['accuracy'].iloc[0] \
    if (subtype_report_small['true_label'] == 'bot').any() else float('nan')

In [ ]:
worst_behav_bot = subtype_report[subtype_report['true_label'] == 'bot'].index[0] \
    if (subtype_report['true_label'] == 'bot').any() else None
worst_behav_acc = subtype_report[subtype_report['true_label'] == 'bot']['accuracy'].iloc[0] \
    if (subtype_report['true_label'] == 'bot').any() else float('nan')

worst_cic = cic_type_report[cic_type_report['is_attack']].iloc[0] if cic_type_report['is_attack'].any() else None
worst_unsw = unsw_type_report[unsw_type_report['is_attack']].iloc[0] if unsw_type_report['is_attack'].any() else None

worst_cic_name = worst_cic['attack_type'] if worst_cic is not None else None
worst_cic_recall = worst_cic['recall'] if worst_cic is not None else 0.0
worst_unsw_name = worst_unsw['attack_type'] if worst_unsw is not None else None
worst_unsw_recall = worst_unsw['recall'] if worst_unsw is not None else 0.0

print(worst_behav_bot, worst_behav_acc)
print(worst_bot_overall, worst_bot_overall_acc)
print(worst_cic_name, worst_cic_recall)
print(worst_unsw_name, worst_unsw_recall)